In [ ]:
import os
DATA_ROOT = os.environ.get('WRH_DATA_ROOT', 'G:/WRH_data')  # local data folder; see README
# ============================================================
# Re-base risk tables: reference = each (model, ssp)'s own 2021–2040
#   tableA(ERA5 baseline) → tableA_ref → tableB_ref → B1_ref / B2_ref
#   coefficients still from M2: panel_cache_M2\grid2\M2_results_summary.csv (unchanged)
#   region×month weights w_acc / w_trf reused from the old tableB (identical to cell 3)
# ============================================================
import os, numpy as np, pandas as pd

O   = rf'{DATA_ROOT}/new_climate/_proj_main'
TB  = os.path.join(O, 'tables')
M2C = rf'{DATA_ROOT}/AIS_processing/panel_cache_M2/grid2'
OUT_TB = TB
REF_SLICE = '2021-2040'
SUFFIX = '_ref2021_2040'

# ── 1. re-base tableA ────────────────────────────────────────
TA = pd.read_parquet(os.path.join(TB, 'tableA_hazard_monthly.parquet'))
key = ['model', 'scenario', 'variable', 'region', 'cmonth', 'metric']
ref = (TA[TA['slice'] == REF_SLICE][key + ['future']].rename(columns={'future': 'base'}))
TA2 = TA.drop(columns=['base', 'delta']).merge(ref, on=key, how='left')
TA2['delta'] = TA2.future - TA2.base
assert TA2.base.notna().all(), 'some (model, scenario, region, cmonth, metric) lack a 2021–2040 reference'
TA2.to_parquet(os.path.join(OUT_TB, f'tableA_hazard_monthly{SUFFIX}.parquet'), index=False)
print('tableA_ref', TA2.shape)

# ── 2. coefficients (M2) and bin → coefficient-name mapping ───────────────────────
S = pd.read_csv(os.path.join(M2C, 'M2_results_summary.csv'))
BETA = {b: {x.label.split('/')[1]: np.log(x.IRR) for _, x in
            S[(S.block == b) & S.label.str.startswith('A_it/W')].iterrows()}
        for b in ('RQ1', 'RQ1-joint')}
COEF_KEY = {
 'D_1.25_2.5': 'Wv_1p25_2p5', 'D_2.5_4': 'Wv_2p5_4', 'D_4_6': 'Wv_4_6', 'D_6_9': 'Wv_6_9', 'D_9_inf': 'Wv_9_p',
 'D_8_10.8': 'Wd_8p0_10p8', 'D_10.8_13.9': 'Wd_10p8_13p9', 'D_13.9_17.2': 'Wd_13p9_17p2',
 'D_17.2_20.8': 'Wd_17p2_20p8', 'D_20.8_inf': 'Wd_20p8_p'}

# ── 3. tableB: delta × log(IRR) → RR ───────────────────────
W = (pd.read_parquet(os.path.join(TB, 'tableB_risk_monthly.parquet'))
       [['region', 'cmonth', 'w_acc', 'w_trf']].drop_duplicates())
d = TA2[TA2.metric.isin(COEF_KEY)].copy()
d['coef_joint'] = d.metric.map(lambda x: BETA['RQ1-joint'].get(COEF_KEY[x], 0.0))
d['coef_rq1']   = d.metric.map(lambda x: BETA['RQ1'].get(COEF_KEY[x], 0.0))
d['lr_joint'] = d.delta * d.coef_joint
d['lr_rq1']   = d.delta * d.coef_rq1
g = d.groupby(['model', 'scenario', 'slice', 'region', 'cmonth', 'variable'])[['lr_joint', 'lr_rq1']].sum().reset_index()
p = g.pivot_table(index=['model', 'scenario', 'slice', 'region', 'cmonth'],
                  columns='variable', values=['lr_joint', 'lr_rq1']).reset_index()
p.columns = ['model', 'scenario', 'slice', 'region', 'cmonth',
             'LR_joint_wave', 'LR_joint_wind', 'LR_rq1_wave', 'LR_rq1_wind']
p['LR_joint']    = p.LR_joint_wave + p.LR_joint_wind
p['RR_joint']    = np.exp(p.LR_joint)
p['RR_rq1_wave'] = np.exp(p.LR_rq1_wave)
p['RR_rq1_wind'] = np.exp(p.LR_rq1_wind)
TB2 = p.merge(W, on=['region', 'cmonth'], how='left')
TB2.to_parquet(os.path.join(OUT_TB, f'tableB_risk_monthly{SUFFIX}.parquet'), index=False)
print('tableB_ref', TB2.shape)

# ── 4. B1 (by region) / B2 (five-region total): annual / seasonal, accident- and traffic-weighted ──
SMAP = {12: 'DJF', 1: 'DJF', 2: 'DJF', 3: 'MAM', 4: 'MAM', 5: 'MAM',
        6: 'JJA', 7: 'JJA', 8: 'JJA', 9: 'SON', 10: 'SON', 11: 'SON'}
def collapse(df, keys, rr, w):
    return (df.assign(_n=df[w] * df[rr], _d=df[w]).groupby(keys, as_index=False)[['_n', '_d']].sum()
              .assign(RR=lambda x: x._n / x._d).drop(columns=['_n', '_d']))
TBs = TB2.assign(season=TB2.cmonth.map(SMAP))
out1, out2 = [], []
for rr in ('RR_joint', 'RR_rq1_wave', 'RR_rq1_wind'):
    for w in ('w_acc', 'w_trf'):
        for lab, extra in (('ANN', []), ('SEA', ['season'])):
            c = collapse(TBs, ['model', 'scenario', 'slice', 'region'] + extra, rr, w)
            c['pct'] = (c.RR - 1) * 100; c['coef'] = rr; c['weight'] = w; c['level'] = lab
            out1.append(c)
            t = collapse(TBs, ['model', 'scenario', 'slice'] + extra, rr, w)
            t['pct'] = (t.RR - 1) * 100; t['MWRI'] = t.RR * 100
            t['coef'] = rr; t['weight'] = w; t['level'] = lab
            out2.append(t)
B1 = pd.concat(out1, ignore_index=True); B2 = pd.concat(out2, ignore_index=True)
B1.to_csv(os.path.join(OUT_TB, f'B1_risk_region{SUFFIX}.csv'), index=False)
B2.to_csv(os.path.join(OUT_TB, f'B2_risk_total{SUFFIX}.csv'), index=False)
print('B1_ref', B1.shape, ' B2_ref', B2.shape)

chk = B2[(B2['slice'] == REF_SLICE)].pct.abs().max()
print(f'reference period max |pct| = {chk:.2e} (should be ≈0)')
q = B1[(B1.level == 'SEA') & (B1.weight == 'w_trf') & (B1['slice'] == '2081-2100')]
print('\n2081–2100 vs 2021–2040, RR_rq1_wave, two-model mean (%):')
print(q[q.coef == 'RR_rq1_wave'].groupby(['region', 'scenario', 'season']).pct.mean()
       .unstack(['season', 'scenario']).round(1).to_string())


In [ ]:
# ============================================================
# Panel b: heat map of change in accident occurrence rate (wave / wind blocks)
#   rows = five regions + All regions; columns = season × scenario; values = two-model mean (%); dots = models agree in sign
#   Inputs: B1_risk_region{SUFFIX}.csv, B2_risk_total{SUFFIX}.csv
# ============================================================
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap

TB     = rf'{DATA_ROOT}/new_climate/_proj_main/tables'
SUFFIX = '_ref2021_2040'          # '' = old ERA5-baseline tables
B1 = pd.read_csv(os.path.join(TB, f'B1_risk_region{SUFFIX}.csv'))
B2 = pd.read_csv(os.path.join(TB, f'B2_risk_total{SUFFIX}.csv'))

# ── Tunable parameters ────────────────────────────────────────────────
SLICE   = '2081-2100'
WEIGHT  = 'w_trf'                 # 'w_trf' traffic weights (original figure) | 'w_acc' accident weights
VMAX    = 8                       # colour bar range ±VMAX (%)
TICKS   = [-8, -4, 0, 4, 8]       # colour bar ticks
CELL_FS = 13                      # font size of numbers in cells
TICK_FS = 13                      # font size of season labels
ROW_FS  = 11.5                    # font size of row labels
PANELS  = [('RR_rq1_wave', 'Wave-related occurence-rate change'),
           ('RR_rq1_wind', 'Wind-related occurence-rate change')]
REGS    = ['NW_Europe', 'NW_Atlantic', 'Med_Black', 'Asia', 'Indian_Ocean_N']
RLAB    = {'NW_Europe': 'NWE', 'NW_Atlantic': 'NWA', 'Med_Black': 'MBS',
           'Asia': 'ESA', 'Indian_Ocean_N': 'NIO', 'All': 'All\nregions'}
SEASONS = ['DJF', 'MAM', 'JJA', 'SON']
SSPS    = ['ssp126', 'ssp585']
SLAB    = {'ssp126': 'SSP1-2.6', 'ssp585': 'SSP5-8.5'}
SCOL    = {'ssp126': '#3b7dd8', 'ssp585': '#c0392b'}
REF_TXT = '2021–2040' if SUFFIX else 'ERA5 2002–2020'
LETTER  = 'b'

CMAP = LinearSegmentedColormap.from_list('BuGyRd',
        ['#0b3d70', '#1b6ba8', '#4a97c9', '#9cc7e0', '#d9e8f2', '#f2f2f0',
         '#fbe3d5', '#f4b79a', '#e07c58', '#c0452c', '#8c1d13'])
INK, INK2 = '#1a1a1a', '#5c6166'
plt.rcParams.update({'font.family': 'sans-serif',
                     'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
                     'axes.unicode_minus': False})

# ── Matrix: mean + sign-agreement flag ────────────────────────────────────
def matrices(coef):
    d1 = B1[(B1.level == 'SEA') & (B1.coef == coef) & (B1.weight == WEIGHT) & (B1['slice'] == SLICE)]
    d2 = B2[(B2.level == 'SEA') & (B2.coef == coef) & (B2.weight == WEIGHT) & (B2['slice'] == SLICE)]
    cols = [(s, p) for s in SEASONS for p in SSPS]
    M = np.full((len(REGS) + 1, len(cols)), np.nan); A = np.zeros_like(M, bool)
    for j, (s, p) in enumerate(cols):
        for i, r in enumerate(REGS):
            v = d1[(d1.region == r) & (d1.season == s) & (d1.scenario == p)].set_index('model').pct
            M[i, j] = v.mean(); A[i, j] = len(v) == 2 and np.sign(v.iloc[0]) == np.sign(v.iloc[1])
        v = d2[(d2.season == s) & (d2.scenario == p)].set_index('model').pct
        M[-1, j] = v.mean(); A[-1, j] = len(v) == 2 and np.sign(v.iloc[0]) == np.sign(v.iloc[1])
    return M, A, cols

def fmt(z):
    r = round(float(z), 1) + 0.0
    return '0.0' if r == 0 else f'{r:+.1f}'.replace('-', '−')

# ── Plot ────────────────────────────────────────────────────
fig = plt.figure(figsize=(15.5, 6.0), dpi=500)
gs = fig.add_gridspec(1, 2, left=.06, right=.985, top=.86, bottom=.26, wspace=.14)
norm = mpl.colors.Normalize(-VMAX, VMAX)

for k, (coef, title) in enumerate(PANELS):
    ax = fig.add_subplot(gs[0, k])
    M, A, cols = matrices(coef)
    nr, nc = M.shape
    ax.set_xlim(-.5, nc - .5); ax.set_ylim(nr - .5, -.5)
    for i in range(nr):
        for j in range(nc):
            z = M[i, j]
            ax.add_patch(Rectangle((j - .43, i - .42), .86, .84, facecolor=CMAP(norm(z)),
                                   edgecolor='none', zorder=2))
            dark = abs(z) > .58 * VMAX
            ax.text(j, i, fmt(z), ha='center', va='center', fontsize=CELL_FS, fontweight='bold',
                    color='white' if dark else '#2b2b2b', zorder=4)
            if A[i, j]:
                ax.plot(j + .33, i - .30, 'o', ms=5, color='#111111', zorder=5)
    for j, (s, p) in enumerate(cols):
        ax.add_patch(Rectangle((j - .47, -.47), .94, nr - .06, fill=False, ls=(0, (3, 2)),
                               lw=1.3, edgecolor=SCOL[p], zorder=3))
    ax.axhline(nr - 1.5, color='#333333', lw=1.6, zorder=3)
    ax.set_xticks([2 * s + .5 for s in range(len(SEASONS))])
    ax.set_xticklabels(SEASONS, fontsize=TICK_FS, fontweight='bold')
    ax.set_yticks(range(nr))
    ax.set_yticklabels([RLAB[r] for r in REGS] + [RLAB['All']], fontsize=ROW_FS, fontweight='bold')
    ax.tick_params(length=0, pad=6)
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_title(title, fontsize=14, fontweight='bold', color=INK, loc='left', pad=10)

fig.text(.018, .93, LETTER, fontsize=17, fontweight='bold', color=INK)

# Colour bar
cax = fig.add_axes([.39, .09, .27, .028])
cb = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=CMAP), cax=cax, orientation='horizontal')
cb.outline.set_visible(False)
cb.set_ticks(TICKS)
cb.set_ticklabels([f'{t:g}'.replace('-', '−') for t in TICKS])
cax.tick_params(labelsize=9.5, length=3, colors=INK2)
cax.set_xlabel('Projected change in accident occurrence rate (%)', fontsize=10.5, color=INK,
               fontweight='bold', labelpad=4)

# Legend
h = [Line2D([0], [0], color=SCOL[p], lw=1.6, ls=(0, (3, 2)), label=SLAB[p]) for p in SSPS]
fig.legend(handles=h, loc='center', bbox_to_anchor=(.19, .11), ncol=2, frameon=False,
           fontsize=11, handlelength=2.6, prop={'weight': 'bold'})
fig.legend(handles=[Line2D([0], [0], marker='o', color='#111111', ms=7, lw=0,
                            label='ACCESS-CM2 and EC-Earth3 agree\non the direction of change')],
           loc='center', bbox_to_anchor=(.83, .11), frameon=False, fontsize=10.5,
           prop={'weight': 'bold'})
fig.text(.06, .015, f'{SLICE} relative to {REF_TXT} · two-model mean · '
         f'{"traffic" if WEIGHT == "w_trf" else "accident"}-weighted within region',
         fontsize=9, color=INK2)
plt.show()
